# Semana 33 · ¿Cuánto brilla una estrella de verdad?

Este cuaderno acompaña a la sesión de proyecto de la semana 33: conversores de magnitudes y distancias, las estrellas más brillantes, la temperatura a partir del índice de color y la curva de luz de una binaria eclipsante.

**Cómo se usa.** Cada recuadro gris es una celda de código: selecciónala y pulsa `Mayús + Intro` para ejecutarla. Ejecuta las celdas **en orden**, porque las de más abajo usan variables definidas en las de arriba. Puedes cambiar cualquier número y volver a ejecutar; no puedes estropear nada.

Tiempo estimado: 30-45 minutos.

## 1. Conversores

$$m_1 - m_2 = -2{,}5\log_{10}\frac{F_1}{F_2}, \qquad m - M = 5\log_{10}\frac{d}{10\ \text{pc}}, \qquad \frac{L}{L_\odot} = 10^{-0{,}4\,(M - 4{,}83)}.$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

PC_EN_AL = 3.2616            # años luz en un pársec
M_SOL = 4.83                 # magnitud absoluta visual del Sol

def cociente_flujo(dm):
    """Cuántas veces más flujo da el objeto más brillante si las magnitudes difieren en dm."""
    return 10 ** (0.4 * dm)

def dif_magnitud(cociente):
    return 2.5 * np.log10(cociente)

def magnitud_absoluta(m, d_pc):
    return m - 5 * np.log10(d_pc / 10)

def distancia_pc(m, M):
    return 10 ** ((m - M + 5) / 5)

def luminosidad_solar(M):
    return 10 ** (-0.4 * (M - M_SOL))

print(f"5 magnitudes = factor {cociente_flujo(5):.0f}; 1 magnitud = factor {cociente_flujo(1):.3f}")
print(f"factor 10 = {dif_magnitud(10):.2f} magnitudes")
print(f"un Sol a 100 pc se vería con m = {M_SOL + 5 * np.log10(100 / 10):.2f}")
print(f"una estrella con m = 12 y M = 2 está a {distancia_pc(12, 2):,.0f} pc".replace(",", " "))

## 2. Las estrellas más brillantes

Magnitudes aparentes visuales y distancias **aproximadas** de algunas de las estrellas más brillantes del cielo. Las distancias de las estrellas lejanas (Rigel, Betelgeuse, Antares, Deneb) tienen incertidumbres grandes; las de Deneb, sobre todo. Betelgeuse y Antares, además, son variables.

In [ ]:
estrellas = [  # nombre, magnitud aparente, distancia en años luz (aproximada)
    ("Sirio", -1.46, 8.6), ("Canopus", -0.74, 310), ("Arturo", -0.05, 37), ("Vega", 0.03, 25),
    ("Capella", 0.08, 43), ("Rigel", 0.13, 860), ("Proción", 0.34, 11.5), ("Betelgeuse", 0.5, 550),
    ("Aldebarán", 0.86, 65), ("Antares", 1.06, 550), ("Deneb", 1.25, 2600), ("Polar", 1.98, 430),
]
print(f"{'estrella':<12}{'m':>6}{'d (pc)':>9}{'M':>7}{'L/L☉':>11}")
datos = []
for nombre, m, d_al in estrellas:
    d = d_al / PC_EN_AL
    M = magnitud_absoluta(m, d)
    L = luminosidad_solar(M)
    datos.append((nombre, m, M, L))
    print(f"{nombre:<12}{m:>6.2f}{d:>9.1f}{M:>7.2f}{L:>11,.0f}".replace(",", " "))

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))
for nombre, m, M, L in datos:
    ax.scatter(m, M, s=40 + 30 * np.log10(L), color="tab:blue")
    ax.annotate(nombre, (m, M), textcoords="offset points", xytext=(6, 4), fontsize=9)
ax.axhline(M_SOL, color="orange", ls="--", label="el Sol (M = 4,83)")
ax.invert_xaxis(); ax.invert_yaxis()      # las magnitudes van al revés: lo brillante, arriba y a la derecha
ax.set_xlabel("magnitud aparente m (lo que vemos)"); ax.set_ylabel("magnitud absoluta M (lo que emiten)")
ax.legend(); ax.grid(alpha=0.3); plt.show()

Todas parecen parecidas en el cielo (de −1,5 a 2), pero sus magnitudes absolutas van de 1,4 (Sirio) a −8 (Deneb): un factor de decenas de miles en luminosidad. Las estrellas más brillantes de nuestro cielo son una mezcla de estrellas cercanas normales y de estrellas lejanas muy luminosas.

**Ejercicio.** Añade el Sol a la lista ($m = -26{,}74$, $d = 1$ ua $= 1/206\,265$ pc) y comprueba que sale $M \approx 4{,}83$.

In [ ]:
# Tu código aquí

In [ ]:
# Solución
print(f"M del Sol = {magnitud_absoluta(-26.74, 1 / 206265):.2f}")

## 3. La temperatura a partir del índice de color

En 2012, el físico Francisco J. Ballesteros publicó una fórmula sencilla que relaciona el índice de color $B - V$ con la temperatura, suponiendo que las estrellas son cuerpos negros:

$$T \approx 4600\left(\frac{1}{0{,}92\,(B-V) + 1{,}7} + \frac{1}{0{,}92\,(B-V) + 0{,}62}\right)\ \text{K}.$$

Es una aproximación: funciona razonablemente para estrellas entre unos 3500 y 10 000 K.

In [ ]:
def temperatura_ballesteros(bv):
    return 4600 * (1 / (0.92 * bv + 1.7) + 1 / (0.92 * bv + 0.62))

for nombre, bv in [("Rigel", -0.03), ("Vega", 0.00), ("Proción", 0.42), ("Sol", 0.65),
                   ("Arturo", 1.23), ("Aldebarán", 1.54), ("Betelgeuse", 1.85)]:
    print(f"{nombre:<11} B−V = {bv:5.2f}  →  T ≈ {temperatura_ballesteros(bv):6.0f} K")

bv = np.linspace(-0.3, 2.0, 200)
plt.figure(figsize=(7, 4))
plt.plot(bv, temperatura_ballesteros(bv))
plt.xlabel("B − V"); plt.ylabel("temperatura (K)"); plt.grid(alpha=0.3); plt.show()

La fórmula da unos 5800 K para el Sol (el valor real es 5772 K) y se desvía para las estrellas muy calientes, como Rigel, cuyo índice cambia muy poco con la temperatura: en el lado azul de la curva de Planck, los dos filtros caen en la zona de pendiente parecida.

## 4. La curva de luz de una binaria eclipsante

Un modelo sencillo de un sistema parecido a Algol: dos estrellas esféricas en órbita circular, vistas casi de canto. La estrella A es pequeña y muy brillante por unidad de superficie; la B, más grande y mucho más tenue. Cuando B pasa por delante de A, tapa buena parte de la luz: es el **mínimo principal**. Media órbita después, A pasa por delante de B y apenas se nota: el **mínimo secundario**.

Para calcular cuánto disco tapa una estrella a otra usamos el área de intersección de dos círculos, una fórmula clásica de geometría. Los parámetros están elegidos a mano para parecerse a Algol; no son un ajuste a datos reales.

In [ ]:
def area_interseccion(r1, r2, d):
    """Área común de dos círculos de radios r1 y r2 con centros a distancia d."""
    d = np.asarray(d, float)
    area = np.zeros_like(d)
    dentro = d <= abs(r1 - r2)
    area[dentro] = np.pi * min(r1, r2) ** 2
    parcial = (d > abs(r1 - r2)) & (d < r1 + r2)
    dd = d[parcial]
    a1 = r1**2 * np.arccos((dd**2 + r1**2 - r2**2) / (2 * dd * r1))
    a2 = r2**2 * np.arccos((dd**2 + r2**2 - r1**2) / (2 * dd * r2))
    a3 = 0.5 * np.sqrt((-dd + r1 + r2) * (dd + r1 - r2) * (dd - r1 + r2) * (dd + r1 + r2))
    area[parcial] = a1 + a2 - a3
    return area

R_A, R_B = 1.0, 1.25        # radios (unidades arbitrarias)
S_A, S_B = 1.0, 0.06        # brillo por unidad de superficie
a, incl = 5.0, np.radians(82)
periodo = 2.867             # días

t = np.linspace(-0.2 * periodo, 1.2 * periodo, 4000)
fase = 2 * np.pi * t / periodo
sep = a * np.sqrt(np.sin(fase) ** 2 + (np.cos(incl) * np.cos(fase)) ** 2)   # separación vista en el cielo
B_delante = np.cos(fase) > 0                                                 # en fase 0, B está delante de A
tapado = area_interseccion(R_A, R_B, sep)
F_A, F_B = S_A * np.pi * R_A**2, S_B * np.pi * R_B**2
flujo = np.where(B_delante, F_A + F_B - S_A * tapado, F_A + F_B - S_B * tapado)
mag = 2.1 - 2.5 * np.log10(flujo / (F_A + F_B))      # fuera de eclipse, magnitud 2,1

fig, (a1, a2) = plt.subplots(2, 1, figsize=(10, 6), sharex=True)
a1.plot(t, flujo / (F_A + F_B)); a1.set_ylabel("flujo relativo"); a1.grid(alpha=0.3)
a2.plot(t, mag); a2.invert_yaxis(); a2.set_ylabel("magnitud"); a2.set_xlabel("tiempo (días)"); a2.grid(alpha=0.3)
plt.suptitle("Curva de luz de una binaria eclipsante (modelo simplificado)"); plt.show()
print(f"magnitud en el mínimo principal: {mag.max():.2f}")
cerca = np.abs(t) < 0.3
en_eclipse = t[cerca][mag[cerca] > 2.15]
print(f"duración del eclipse principal: {(en_eclipse.max() - en_eclipse.min()) * 24:.1f} horas")

Las mismas observaciones en flujo y en magnitudes: la curva en magnitudes va al revés (por eso invertimos el eje). En el mínimo, el sistema baja más de una magnitud, más o menos como Algol, y el eclipse dura unas horas.

## 5. Experimenta

1. Cambia la inclinación a 75°. ¿Qué pasa con la profundidad del mínimo? ¿Y a 90°?
2. Haz las dos estrellas iguales (mismo radio y mismo brillo por superficie). ¿Cómo son los dos mínimos?
3. Con los conversores, calcula hasta qué distancia verías a simple vista (magnitud 6) a Sirio, a Rigel y al Sol.

In [ ]:
# Tu experimento aquí